# 📊 Base de Datos — Trabajo Final de Grado
## Riesgo País Argentino: Series Temporales

**Autor:** Martín  
**Universidad:** Universidad Nacional de Córdoba  
**Licenciatura en Economía**

---

### Guía rápida de uso

| Sección | Qué hace |
|---------|----------|
| **0. Setup** | Importaciones y configuración inicial |
| **1. Glosario** | Ver todas las series disponibles, filtrar por fuente o categoría |
| **2. Serie individual** | Elegir fuente + serie, horizonte temporal y frecuencia → gráfico |
| **3. Comparador** | Elegir 2+ series → base 100 en escala log + diferencia |
| **4. Test ADF/KPSS** | Test de estacionariedad sobre las series del comparador |
| **5. HTML interactivo** | Genera `visualizador.html` con menú desplegable (standalone) |
| **6. Exportación** | Guarda el dataset consolidado en CSV |

## 0. Configuración e Importaciones

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
import os
import sys

sys.path.append(os.path.abspath('..'))
from src.glosario import GLOSARIO, buscar, cargar_serie, info, get as _get_meta
from src.utils import test_estacionariedad

warnings.filterwarnings('ignore')
try:
    plt.style.use('seaborn-v0_8-darkgrid')
except:
    try:
        plt.style.use('seaborn-darkgrid')
    except:
        pass  # Usa estilo por defecto

# Resumen del glosario
total = len(GLOSARIO)
disponibles = sum(1 for e in GLOSARIO if e.get('disponible', True))
print(f"✅ Glosario cargado: {total} series en total")
print(f"   → {disponibles} disponibles | {total - disponibles} pendientes de descarga")
print(f"\nFuentes: {sorted(set(e['fuente'] for e in GLOSARIO))}")
print(f"Categorías: {sorted(set(e['categoria'] for e in GLOSARIO))}")

✅ Glosario cargado: 90 series en total
   → 87 disponibles | 3 pendientes de descarga

Fuentes: ['bcra', 'fred', 'global', 'indec', 'local', 'mecon', 'mep', 'processed', 'worldbank']
Categorías: ['cambiario', 'commodity', 'dependiente', 'dummy', 'externo', 'fiscal', 'institucional', 'mercado', 'monetario', 'real']


---
## 1. Glosario de Series Disponibles

Filtrá por `fuente` o `categoria`. Dejá vacío para ver todas.

**Fuentes disponibles:** `global`, `fred`, `bcra`, `local`, `indec`, `worldbank`, `mecon`, `processed`, `mep`  
**Categorías:** `dependiente`, `mercado`, `cambiario`, `commodity`, `monetario`, `externo`, `fiscal`, `real`, `institucional`, `dummy`  
La columna `disponible` indica si el archivo CSV existe en disco (⚠️ = pendiente de descarga).

In [2]:
# ============================================================
#  MODIFICAR AQUÍ — dejá '' para ver todas las series
# ============================================================
FILTRO_FUENTE    = ''   # ej: 'bcra', 'global', 'indec', 'worldbank', 'mecon'
FILTRO_CATEGORIA = ''   # ej: 'mercado', 'fiscal', 'cambiario', 'monetario'
FILTRO_TEXTO     = ''   # búsqueda libre en nombre/descripción/id

df_glosario = buscar(texto=FILTRO_TEXTO, fuente=FILTRO_FUENTE, categoria=FILTRO_CATEGORIA)

pd.set_option('display.max_colwidth', 60)
pd.set_option('display.max_rows', 200)
print(f"Series encontradas: {len(df_glosario)}")
display(df_glosario.reset_index(drop=True))

Series encontradas: 90


,id,nombre,fuente,categoria,frecuencia,unidad,desde,hasta,descripcion,disponible
0,embi_arg,EMBI+ Argentina,global,dependiente,diaria,puntos básicos,1999-01-22,2026-04-24,Riesgo país argentino medido por el EMBI+ de J.P. Morgan,✅
1,vix,VIX,global,mercado,diaria,puntos,2000-01-03,2026-04-22,Índice de volatilidad implícita del S&P 500 (CBOE),✅
2,move,MOVE Index,global,mercado,diaria,puntos básicos,2002-11-12,2026-04-22,ICE BofA MOVE Index — volatilidad implícita del mercado ...,✅
3,dxy,DXY (Índice del Dólar),global,cambiario,diaria,índice,2000-01-03,2026-04-22,Índice del dólar estadounidense contra una canasta de mo...,✅
4,sp500,S&P 500,global,mercado,diaria,puntos,2000-01-03,2026-04-22,Índice bursátil S&P 500,✅
5,msci_em,MSCI Emergentes,global,mercado,diaria,puntos,2003-04-14,2026-04-22,Índice MSCI de mercados emergentes,✅
6,embi_latam,EMBI Latinoamérica,global,mercado,diaria,puntos básicos,2007-10-29,2026-04-22,Spread EMBI promedio de países latinoamericanos,✅
7,ofr_fsi,OFR Financial Stress Index,global,mercado,diaria,índice estandarizado,2000-01-03,2026-04-22,Índice de estrés financiero de la Oficina de Investigaci...,✅
8,ust2y,UST 2Y,global,mercado,diaria,% anual,2000-01-03,2026-04-22,Rendimiento del bono del Tesoro de EEUU a 2 años,✅
9,ust5y,UST 5Y,global,mercado,diaria,% anual,2000-01-03,2026-04-22,Rendimiento del bono del Tesoro de EEUU a 5 años,✅


---
## 2. Visualización de Serie Individual

**Cómo usar:**
1. Copiá el `id` de la serie de la tabla del glosario de arriba
2. Pegalo en `SERIE_ID`
3. Ajustá las fechas y la frecuencia
4. Re-ejecutá la celda

> **Frecuencias disponibles:**
> - `None` → frecuencia original del archivo
> - `'D'` → diaria | `'W'` → semanal | `'M'` → mensual | `'Q'` → trimestral | `'A'` → anual
> 
> ⚠️ **No podés subir la frecuencia** (si la serie es mensual, no podés pedir diaria). Sí podés bajarla (de diaria a mensual).

In [3]:
# ============================================================
#  CONFIGURACIÓN — MODIFICAR AQUÍ
# ============================================================
SERIE_ID     = 'embi_arg'    # ID del glosario (copiarlo de la tabla de arriba)
FECHA_INICIO = '2003-01-01'  # Horizonte temporal inicio 'YYYY-MM-DD'
FECHA_FIN    = pd.Timestamp.today().strftime('%Y-%m-%d')  # Hasta hoy

# Frecuencia de resampleo:
#   None = original del archivo
#   'D'=diaria | 'W'=semanal | 'M'=mensual | 'Q'=trimestral | 'A'=anual
FRECUENCIA   = None

# ============================================================
# Mostrar ficha de la serie
info(SERIE_ID)

# Verificar frecuencias compatibles
meta = _get_meta(SERIE_ID)
if meta:
    freq_orden = {'diaria': 1, 'semanal': 2, 'mensual': 3, 'trimestral': 4, 'anual': 5}
    freq_solicitada_map = {'D': 1, 'W': 2, 'M': 3, 'Q': 4, 'A': 5}
    orig_level = freq_orden.get(meta.get('frecuencia', 'diaria'), 1)
    req_level  = freq_solicitada_map.get(FRECUENCIA, 0) if FRECUENCIA else 0

    print("\n📋 Frecuencias de resampleo disponibles para esta serie:")
    freqs_disp = ['D', 'W', 'M', 'Q', 'A']
    nombres_freq = {'D': 'Diaria', 'W': 'Semanal', 'M': 'Mensual', 'Q': 'Trimestral', 'A': 'Anual'}
    for f in freqs_disp:
        nivel = freq_solicitada_map[f]
        if nivel >= orig_level:
            marca = '✅' if f != FRECUENCIA else '👉 (seleccionada)'
        else:
            marca = '❌ (no disponible — frecuencia más alta que la original)'
        print(f"   '{f}' — {nombres_freq[f]} {marca}")
    print(f"   None — Original ({meta.get('frecuencia', '?')}) {'👉 (seleccionada)' if FRECUENCIA is None else ''}")

    if req_level > 0 and req_level < orig_level:
        print(f"\n❌ ERROR: No podés subir la frecuencia de '{meta['frecuencia']}' a '{FRECUENCIA}'. Cambiá FRECUENCIA.")
        serie = None
    else:
        # Cargar la serie
        serie = cargar_serie(SERIE_ID, FECHA_INICIO, FECHA_FIN, FRECUENCIA)
else:
    serie = None

if serie is None:
    print("\n⚠️ Serie no disponible con la configuración actual.")
else:
    freq_str = FRECUENCIA or meta.get('frecuencia', 'original')
    print(f"\n📊 {len(serie)} observaciones: {serie.index.min().date()} → {serie.index.max().date()}")
    print(f"   Frecuencia cargada: {freq_str}")

    # --- Gráfico Plotly ---
    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=serie.index, y=serie.values,
        mode='lines',
        name=meta['nombre'],
        line=dict(color='#00b4d8', width=1.8),
        hovertemplate='%{x|%Y-%m-%d}<br><b>%{y:.4f}</b><extra></extra>'
    ))
    fig.update_layout(
        title=dict(
            text=f"<b>{meta['nombre']}</b><br>"
                 f"<sup>{meta['descripcion']} | {FECHA_INICIO} → {FECHA_FIN} | Frec: {freq_str}</sup>",
            font=dict(size=15)
        ),
        xaxis=dict(
            title='Fecha',
            rangeslider=dict(visible=True),
            type='date',
            rangeselector=dict(
                buttons=[
                    dict(count=1,  label='1A',  step='year',  stepmode='backward'),
                    dict(count=3,  label='3A',  step='year',  stepmode='backward'),
                    dict(count=5,  label='5A',  step='year',  stepmode='backward'),
                    dict(count=10, label='10A', step='year',  stepmode='backward'),
                    dict(step='all', label='Todo'),
                ]
            )
        ),
        yaxis_title=f"{meta['nombre']} ({meta['unidad']})",
        template='plotly_dark',
        height=500,
        hovermode='x unified',
        annotations=[{
            'text': f"Fuente: {meta['fuente'].upper()} | {meta['unidad']}",
            'xref': 'paper', 'yref': 'paper',
            'x': 0.0, 'y': -0.18,
            'showarrow': False,
            'font': {'size': 10, 'color': '#888'}
        }]
    )
    fig.show()

    # Estadísticas descriptivas
    print(f"\n📈 Estadísticas descriptivas")
    print(serie.describe().round(4).to_string())

  EMBI+ Argentina
  id              embi_arg
  nombre          EMBI+ Argentina
  fuente          global
  categoria       dependiente
  descripcion     Riesgo país argentino medido por el EMBI+ de J.P. Morgan
  frecuencia      diaria
  unidad          puntos básicos
  desde           1999-01-22
  hasta           2026-04-24
  archivo         data/raw/global/riesgo_pais_arg.csv

📋 Frecuencias de resampleo disponibles para esta serie:
   'D' — Diaria ✅
   'W' — Semanal ✅
   'M' — Mensual ✅
   'Q' — Trimestral ✅
   'A' — Anual ✅
   None — Original (diaria) 👉 (seleccionada)

📊 6585 observaciones: 2003-01-02 → 2026-04-22
   Frecuencia cargada: diaria



📈 Estadísticas descriptivas
count    6585.0000
mean     1509.2067
std      1452.8522
min       184.0000
25%       571.0000
50%       900.0000
75%      1864.0000
max      7004.0000


---
## 3. Comparador de Series (Base 100 + Escala Logarítmica)

**Cómo usar:**
1. Listá los IDs en `SERIES_IDS` (mínimo 2)
2. Ajustá el período, la frecuencia y la fecha base de indexación
3. El gráfico superior muestra las series indexadas a base 100 en escala log
4. El gráfico inferior muestra la diferencia entre cada serie y la primera de referencia

In [8]:
# ============================================================
#  CONFIGURACIÓN — MODIFICAR AQUÍ
# ============================================================
SERIES_IDS = [    # Variable dependiente — referencia (base 100)
    'dxy',  # EMBI regional
    'vix',         # Riesgo global
]

FECHA_INICIO_COMP = '2007-01-01'   # Fecha inicio del período de comparación
FECHA_FIN_COMP    = pd.Timestamp.today().strftime('%Y-%m-%d')

# Frecuencia para armonizar las series:
# 'M' suele ser lo mejor para series de distinta frecuencia original
FRECUENCIA_COMP   = 'M'

# ============================================================
# Cargar todas las series
series_comp = {}
for sid in SERIES_IDS:
    s = cargar_serie(sid, FECHA_INICIO_COMP, FECHA_FIN_COMP, FRECUENCIA_COMP)
    if s is not None:
        series_comp[sid] = s
        meta_s = _get_meta(sid)
        print(f"✅  {meta_s['nombre']:45s} | {len(s):5d} obs | {s.index.min().date()} → {s.index.max().date()}")
    else:
        print(f"❌  {sid}: no disponible")

if len(series_comp) < 2:
    print("\n⚠️  Se necesitan al menos 2 series válidas para comparar.")
else:
    # Construir DataFrame consolidado
    df_comp = pd.DataFrame(series_comp)
    df_comp = df_comp.dropna(how='all')

    # Fecha de inicio efectiva
    fecha_base = df_comp.dropna(thresh=1).index.min()

    # Normalizar a base 100 en la fecha de inicio de cada serie
    df_base100 = pd.DataFrame(index=df_comp.index)
    for sid in series_comp:
        col = df_comp[sid].dropna()
        if col.empty:
            continue
        val_base = col.iloc[0]
        if val_base == 0 or pd.isna(val_base):
            continue
        df_base100[sid] = (df_comp[sid] / val_base) * 100

    # Diferencia respecto a la primera serie de referencia
    sid_ref = list(series_comp.keys())[0]
    df_diff = pd.DataFrame(index=df_comp.index)
    for sid in list(series_comp.keys())[1:]:
        if sid in df_base100.columns and sid_ref in df_base100.columns:
            df_diff[f"{sid} − {sid_ref}"] = df_base100[sid] - df_base100[sid_ref]

    # Nombres legibles
    nombres = {sid: _get_meta(sid)['nombre'] for sid in series_comp}
    colores  = px.colors.qualitative.Bold

    fig = make_subplots(
        rows=2, cols=1,
        shared_xaxes=True,
        vertical_spacing=0.07,
        row_heights=[0.65, 0.35],
        subplot_titles=[
            f'Índice Base 100 al {fecha_base.strftime("%Y-%m-%d")} — Escala Logarítmica',
            f'Diferencia vs. {nombres[sid_ref]} (en puntos base 100)'
        ]
    )

    # Gráfico superior: base 100 + log
    for i, sid in enumerate(series_comp):
        if sid not in df_base100.columns:
            continue
        col = df_base100[sid].dropna()
        fig.add_trace(
            go.Scatter(
                x=col.index, y=col.values,
                mode='lines',
                name=nombres[sid],
                line=dict(color=colores[i % len(colores)], width=2),
                hovertemplate=f'<b>{nombres[sid]}</b><br>%{{x|%Y-%m-%d}}<br>Base 100: %{{y:.2f}}<extra></extra>'
            ),
            row=1, col=1
        )

    # Gráfico inferior: diferencias
    for i, col_name in enumerate(df_diff.columns):
        col = df_diff[col_name].dropna()
        fig.add_trace(
            go.Scatter(
                x=col.index, y=col.values,
                mode='lines',
                name=col_name,
                line=dict(color=colores[(i + 1) % len(colores)], width=1.5),
                showlegend=True,
                hovertemplate=f'%{{x|%Y-%m-%d}}<br>Diferencia: %{{y:.2f}}<extra></extra>'
            ),
            row=2, col=1
        )

    # Línea cero
    fig.add_hline(y=0, line=dict(color='white', width=0.8, dash='dot'), row=2, col=1)

    fig.update_yaxes(type='log', row=1, col=1, title_text='Índice (base 100, escala log)')
    fig.update_yaxes(row=2, col=1, title_text='Diferencia (puntos base 100)')
    fig.update_xaxes(row=2, col=1, title_text='Fecha')

    fig.update_layout(
        template='plotly_dark',
        height=780,
        hovermode='x unified',
        legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1)
    )
    fig.show()

    # Correlaciones
    print(f"\n📊 Correlaciones (Pearson) en el período seleccionado:")
    corr = df_comp.corr().round(3)
    corr.columns = [nombres.get(c, c) for c in corr.columns]
    corr.index   = [nombres.get(c, c) for c in corr.index]
    display(corr)

✅  DXY (Índice del Dólar)                        |   232 obs | 2007-01-31 → 2026-04-30
✅  VIX                                           |   232 obs | 2007-01-31 → 2026-04-30



📊 Correlaciones (Pearson) en el período seleccionado:


,DXY (Índice del Dólar),VIX
DXY (Índice del Dólar),1.000,-0.149
VIX,-0.149,1.000


---
## 4. Test de Estacionariedad (ADF / KPSS)

In [5]:
# ============================================================
#  Series a testear (usa las del comparador, o definí otras)
# ============================================================
SERIES_TEST = SERIES_IDS  # o ej: ['embi_arg', 'vix', 'tc_mayorista']

resultados = []
for sid in SERIES_TEST:
    s = cargar_serie(sid, '2000-01-01', None, 'M')  # mensual para ADF
    if s is None:
        continue
    nombre = _get_meta(sid)['nombre']
    try:
        r = test_estacionariedad(s.dropna(), nombre)
        resultados.append(r)
    except Exception as e:
        print(f"Error en {sid}: {e}")

if resultados:
    print("\n📋 Resultados Tests de Estacionariedad (ADF / KPSS)")
    print('='*70)
    display(pd.DataFrame(resultados))
else:
    print("No hay resultados — verificá que las series del Paso 3 estén disponibles.")

Error en embi_arg: No module named 'statsmodels'
Error en embi_latam: No module named 'statsmodels'
Error en vix: No module named 'statsmodels'
No hay resultados — verificá que las series del Paso 3 estén disponibles.


---
## 5. Generar HTML Interactivo

Esta celda genera `visualizador.html` — un archivo standalone que podés abrir en cualquier browser sin Python.

Incluye:
- **Menú desplegable** por fuente y por serie
- **Selector de rango** de fechas (1A, 3A, 5A, 10A, Todo)
- **Información** de cada serie al seleccionarla

In [6]:
# ============================================================
#  Configuración del HTML
# ============================================================
# Series a incluir: None = todas las disponibles del glosario
SERIES_HTML = None  # o ej: ['embi_arg', 'vix', 'sp500', 'tc_mayorista', 'merval']

FREQ_HTML   = 'M'     # Frecuencia de carga ('M' = mensual es el mejor balance)
OUTPUT_HTML = os.path.join('..', 'notebooks', 'visualizador.html')

# ============================================================
print("Cargando series para el HTML...")

# Solo incluir series disponibles
ids_html = SERIES_HTML or [e['id'] for e in GLOSARIO if e.get('disponible', True)]

datos_html = {}   # id -> pd.Series
for sid in ids_html:
    s = cargar_serie(sid, '2000-01-01', None, FREQ_HTML)
    if s is not None and not s.empty:
        datos_html[sid] = s
        print(f"  ✅ {sid:35s} {len(s):5d} obs")
    else:
        print(f"  ⚠️  {sid:35s} sin datos")

print(f"\nTotal series cargadas: {len(datos_html)} / {len(ids_html)}")

if not datos_html:
    print("❌ No hay datos para generar el HTML. Verificá las rutas del glosario.")
else:
    # Construir figura Plotly
    colores_html = px.colors.qualitative.Bold + px.colors.qualitative.Pastel + px.colors.qualitative.Safe
    fig_html = go.Figure()

    ids_ordenados = sorted(
        datos_html.keys(),
        key=lambda x: _get_meta(x)['fuente'] + _get_meta(x)['nombre']
    )

    for i, sid in enumerate(ids_ordenados):
        meta_h = _get_meta(sid)
        s = datos_html[sid]
        fig_html.add_trace(go.Scatter(
            x=s.index,
            y=s.values,
            mode='lines',
            name=meta_h['nombre'],
            visible=(i == 0),
            line=dict(color=colores_html[i % len(colores_html)], width=1.8),
            hovertemplate=(
                f'<b>{meta_h["nombre"]}</b><br>'
                '%{x|%Y-%m-%d}<br>'
                f'%{{y:.4f}} {meta_h["unidad"]}'
                '<extra></extra>'
            )
        ))

    # Crear botones del dropdown
    botones = []
    for i, sid in enumerate(ids_ordenados):
        meta_h = _get_meta(sid)
        visibilidad = [j == i for j in range(len(ids_ordenados))]
        s = datos_html[sid]
        botones.append(dict(
            label=f"[{meta_h['fuente'].upper()}] {meta_h['nombre']}",
            method='update',
            args=[
                {'visible': visibilidad},
                {
                    'title.text': f"<b>{meta_h['nombre']}</b> — {meta_h['descripcion']}",
                    'yaxis.title.text': f"{meta_h['nombre']} ({meta_h['unidad']})",
                    'annotations': [{
                        'text': (
                            f"Fuente: {meta_h['fuente'].upper()} | "
                            f"Categoría: {meta_h['categoria']} | "
                            f"Frec: {meta_h['frecuencia']} → resampleada a mensual | "
                            f"{s.index.min().strftime('%Y-%m-%d')} → {s.index.max().strftime('%Y-%m-%d')}"
                        ),
                        'xref': 'paper', 'yref': 'paper',
                        'x': 0, 'y': -0.13,
                        'showarrow': False,
                        'font': {'size': 10, 'color': '#999'}
                    }]
                }
            ]
        ))

    meta_0 = _get_meta(ids_ordenados[0])
    s_0    = datos_html[ids_ordenados[0]]

    fig_html.update_layout(
        updatemenus=[dict(
            buttons=botones,
            direction='down',
            pad={'r': 10, 't': 10},
            showactive=True,
            x=0.0, xanchor='left',
            y=1.16, yanchor='top',
            bgcolor='#1e1e2e',
            font=dict(color='white', size=12),
            bordercolor='#444',
        )],
        title=dict(
            text=f"<b>{meta_0['nombre']}</b> — {meta_0['descripcion']}",
            font=dict(size=14)
        ),
        xaxis=dict(
            rangeslider=dict(visible=True),
            type='date',
            rangeselector=dict(
                buttons=[
                    dict(count=1,  label='1A',  step='year', stepmode='backward'),
                    dict(count=3,  label='3A',  step='year', stepmode='backward'),
                    dict(count=5,  label='5A',  step='year', stepmode='backward'),
                    dict(count=10, label='10A', step='year', stepmode='backward'),
                    dict(step='all', label='Todo'),
                ]
            )
        ),
        yaxis=dict(title=f"{meta_0['nombre']} ({meta_0['unidad']})"),
        template='plotly_dark',
        height=620,
        hovermode='x unified',
        margin=dict(l=60, r=40, t=140, b=90),
        annotations=[{
            'text': (
                f"Fuente: {meta_0['fuente'].upper()} | "
                f"Categoría: {meta_0['categoria']} | "
                f"Frec: {meta_0['frecuencia']} → resampleada a mensual | "
                f"{s_0.index.min().strftime('%Y-%m-%d')} → {s_0.index.max().strftime('%Y-%m-%d')}"
            ),
            'xref': 'paper', 'yref': 'paper',
            'x': 0, 'y': -0.13,
            'showarrow': False,
            'font': {'size': 10, 'color': '#999'}
        }]
    )

    import plotly.io as pio
    pio.write_html(
        fig_html,
        file=OUTPUT_HTML,
        include_plotlyjs=True,
        full_html=True,
        config={'displayModeBar': True, 'scrollZoom': True}
    )
    print(f"\n✅  HTML generado: {os.path.abspath(OUTPUT_HTML)}")
    print(f"    Abrí el archivo en tu browser para explorar las {len(datos_html)} series interactivamente.")

Cargando series para el HTML...
  ✅ embi_arg                              316 obs
  ✅ vix                                   316 obs
  ✅ move                                  282 obs
  ✅ dxy                                   316 obs
  ✅ sp500                                 316 obs
  ✅ msci_em                               277 obs
  ✅ embi_latam                            223 obs
  ✅ ofr_fsi                               316 obs
  ✅ ust2y                                 316 obs
  ✅ ust5y                                 316 obs
  ✅ ust10y                                316 obs
  ✅ ust30y                                316 obs
  ✅ brl                                   269 obs
  ✅ clp                                   269 obs
  ✅ mxn                                   269 obs
  ✅ wti                                   309 obs
  ✅ brent                                 226 obs
  ✅ soja                                  308 obs
  ✅ maiz                                  310 obs
  ✅ trigo         

---
## 6. Exportación de Datos

Genera el dataset consolidado en `data/processed/series_consolidadas.csv` para usar en los modelos econométricos.

In [7]:
# ============================================================
#  Series a exportar y frecuencia del dataset consolidado
# ============================================================
SERIES_EXPORTAR = [
    # --- Variable dependiente ---
    'embi_arg',
    # --- Variables globales ---
    'vix',
    'embi_latam',
    'ust10y',
    'dxy',
    'sp500',
    # --- Commodities ---
    'soja',
    'wti',
    # --- Cambiarias ---
    'tc_mayorista',
    'itcrm',
    # --- Monetarias / BCRA ---
    'reservas_brutas',
    'tasa_badlar',
    'base_monetaria',
    # --- Real ---
    'ipc_var_mensual',
    'emae',
    # --- Fiscal ---
    'resultado_primario',
    # --- Variables procesadas ---
    'spread_tasas',
    'years_since_default',
    'dummy_cepo',
    'dummy_electoral',
]
FRECUENCIA_EXPORT = 'M'    # 'D', 'M', 'Q'
FECHA_INICIO_EXP  = '2000-01-01'

# ============================================================
output_dir = os.path.join('..', 'data', 'processed')
os.makedirs(output_dir, exist_ok=True)

df_export = pd.DataFrame()
errores = []
for sid in SERIES_EXPORTAR:
    s = cargar_serie(sid, FECHA_INICIO_EXP, None, FRECUENCIA_EXPORT)
    if s is not None:
        df_export[sid] = s
        print(f"  ✅ {sid}")
    else:
        errores.append(sid)
        print(f"  ⚠️  {sid}: sin datos")

output_path = os.path.join(output_dir, 'series_consolidadas.csv')
df_export.to_csv(output_path)
print(f"\n✅  Dataset consolidado: {df_export.shape[0]} filas × {df_export.shape[1]} columnas")
print(f"    Guardado en: {output_path}")
if errores:
    print(f"\n⚠️  Series no disponibles: {errores}")
print(f"\nNaN por columna:")
print(df_export.isna().sum().to_string())

  ✅ embi_arg
  ✅ vix
  ✅ embi_latam
  ✅ ust10y
  ✅ dxy
  ✅ sp500
  ✅ soja
  ✅ wti
  ✅ tc_mayorista
  ✅ itcrm
  ✅ reservas_brutas
  ✅ tasa_badlar
  ✅ base_monetaria
  ✅ ipc_var_mensual
  ✅ emae
  ✅ resultado_primario
  ✅ spread_tasas
  ✅ years_since_default
  ✅ dummy_cepo
  ✅ dummy_electoral

✅  Dataset consolidado: 316 filas × 20 columnas
    Guardado en: ..\data\processed\series_consolidadas.csv

NaN por columna:
embi_arg                 0
vix                      0
embi_latam              93
ust10y                   0
dxy                      0
sp500                    0
soja                     8
wti                      7
tc_mayorista            36
itcrm                    0
reservas_brutas         36
tasa_badlar             36
base_monetaria          36
ipc_var_mensual        205
emae                    50
resultado_primario     193
spread_tasas            36
years_since_default      0
dummy_cepo               0
dummy_electoral          0
